# Alineacion de Preferencias en LLMs con DPO (Direct Preference Optimization)

**Nivel:** Avanzado  
**Tecnologias:** Hugging Face `trl` (`DPOTrainer`), `peft` (LoRA), `transformers`  
**Modelo Base:** Google Gemma 2 2B Instruct (`google/gemma-2-2b-it`) / Qwen 2.5 1.5B  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jggomez/workshop-open-models/blob/main/session-03-fine-tuning-llms/02-preference-alignment-dpo/02_preference_alignment_dpo.ipynb)

---

## 1. Fundamentos Teoricos: Alineacion por Preferencias y la Revolucion DPO

### Por que el SFT no es suficiente?
El Fine-Tuning Supervisado (SFT) entrena al modelo mediante maxima verosimilitud para imitar las respuestas del dataset. Sin embargo, tiene dos grandes limitaciones:
1. **Imposibilidad de penalizar comportamientos indeseados:** En SFT, solo mostramos lo que el modelo *debe* decir, pero no tenemos forma de decirle *"esto que generaste es incorrecto o alucinado"*.
2. **Alineacion fina de estilo:** Cuando hay multiples formas de responder una pregunta, el SFT no puede aprender de manera robusta cual alternativa es cualitativamente superior.

### De RLHF clasico a DPO (Direct Preference Optimization)
Tradicionalmente, la alineacion requeria **RLHF con PPO (Proximal Policy Optimization)**:
- Entrenar un modelo de recompensa (*Reward Model*) a partir de comparaciones humanas.
- Optimizar la politica del LLM con aprendizaje por refuerzo continuo frente al Reward Model.
- Mantener en VRAM 4 redes neuronales simultaneas (Actor, Critic, Reward Model y Reference Model).

En 2023, investigadores de Stanford (Rafailov et al.) publicaron **DPO**, demostrando que la funcion de recompensa de RLHF se puede derivar directamente de las razones de probabilidad logaritmica (*log-likelihood ratios*) de la politica actual $\pi_\theta$ frente a una politica de referencia congelada $\pi_{ref}$:

$$\mathcal{L}_{DPO}(\pi_\theta; \pi_{ref}) = -\mathbb{E}_{(x, y_w, y_l)} \left[ \log \sigma \left( \beta \log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \beta \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)} \right) \right]$$

donde:
- $x$ es el prompt.
- $y_w$ (*winner / chosen*) es la respuesta preferida, precisa y sin alucinaciones.
- $y_l$ (*loser / rejected*) es la respuesta indeseada, confusa o alucinada.
- $\beta$ es el hiperparametro de control (generalmente entre $0.05$ y $0.2$) que penaliza la divergencia KL para no olvidar las capacidades generales del modelo base.

### Ventaja en Recursos (PEFT + DPO)
Al usar **PEFT/LoRA**, el modelo base congelado actua implicitamente como $\pi_{ref}$ cuando los adaptadores estan desactivados, y como $\pi_\theta$ cuando estan activos. Esto permite ejecutar DPO en **una unica GPU modesta (Colab T4 de 16 GB)** sin necesidad de cargar dos modelos completos en memoria.


### Paso 1: Instalacion de Bibliotecas Especializadas

Instalamos el stack moderno de Hugging Face con soporte nativo de DPO:


### Configuración fuera de Google Colab

Si estás ejecutando este notebook en un entorno local, servidor propio o contenedor Docker, asegúrate de tener instaladas las siguientes bibliotecas base. A diferencia de Colab, estos entornos suelen estar vacíos:

*   **Motor de Deep Learning:** `torch` y `torchvision` (asegúrate de instalar la versión compatible con tu versión de CUDA).
*   **Ecosistema Hugging Face:** `transformers`, `accelerate` y `datasets`.
*   **Fine-Tuning y Optimización:** `peft` (para LoRA), `trl` (para el SFTTrainer) y `bitsandbytes` (para cuantización de 4/8 bits).

**Comando de instalación recomendado:**
```bash
pip install torch torchvision transformers accelerate datasets peft trl bitsandbytes
```

*Nota: Se recomienda el uso de entornos virtuales (`venv` o `conda`) para evitar conflictos de dependencias.*

In [1]:
!pip install -qU peft trl bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 30.4 MB/s eta 0:00:00


### Paso 2: Importacion de Modulos y Verificacion del Entorno

Importamos `DPOTrainer` y `DPOConfig` de la biblioteca `trl`:


In [2]:
import torch
import transformers
import trl
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import gc

print(f"PyTorch:      {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"TRL:          {trl.__version__}")
print(f"CUDA activa:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo:  {torch.cuda.get_device_name(0)}")

PyTorch:      2.11.0+cu128
Transformers: 5.16.1
TRL:          1.12.0
CUDA activa:  True
Dispositivo:  Tesla T4


### Paso 3: Carga del Modelo Base y Tokenizador con Cuantizacion de 4 Bits

Cargamos `google/gemma-2-2b-it` cuantizado con `BitsAndBytesConfig` para maximizar la memoria disponible para el calculo de log-probabilidades comparativas:


In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # Padding a la izquierda es estandar para generacion y DPO

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto" if torch.cuda.is_available() else None,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    trust_remote_code=True
)

print("Modelo base cargado para alineacion DPO.")

config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Modelo base cargado para alineacion DPO.


### Paso 4: Construccion del Dataset de Preferencias Ternarias (Prompt, Chosen, Rejected)

A diferencia de SFT, en DPO el dataset requiere tres columnas:
- `prompt`: La instruccion formulada por el usuario.
- `chosen`: La respuesta ideal (concisa, verificable, segura, sin rodeos ni alucinaciones).
- `rejected`: La respuesta defectuosa (alucinada, redundante, insegura o con suposiciones falsas).


In [4]:
preference_data = [
    {
        "prompt": "Como cancelo mi suscripcion y obtengo mi factura del mes en curso?",
        "chosen": "Para cancelar su suscripcion y descargar su factura: ingrese a su Perfil > Facturacion, seleccione Cancelar Plan y luego haga clic en Descargar PDF junto al periodo actual. La cancelacion se hace efectiva al final de su ciclo de facturacion.",
        "rejected": "No estoy totalmente seguro, pero creo que tienes que enviar un correo electronico a soporte tecnico o llamar por telefono a nuestras oficinas centrales en California, o quizas cancelar la tarjeta de credito en tu banco para que no te sigan cobrando."
    },
    {
        "prompt": "Cual es la velocidad maxima garantizada de la API en el plan Pro?",
        "chosen": "El plan Pro garantiza un SLA de latencia menor a 120 ms para el 99% de las peticiones (p99) con un limite de tasa de 500 solicitudes por minuto.",
        "rejected": "Nuestra API es la mas rapida del universo entero. Te garantizamos velocidad cuantica instantanea de 0 milisegundos sin limite alguno porque nuestros servidores usan tecnologia alienigena super avanzada."
    },
    {
        "prompt": "Que debo hacer si olvido mi contrasena y la autenticacion 2FA esta bloqueada?",
        "chosen": "Si perdio el acceso a su 2FA: utilice una de sus 8 claves de recuperacion de emergencia descargadas durante el registro. Si no cuenta con ellas, inicie una solicitud de verificacion de identidad en soporte@techcloud.io con su documento oficial.",
        "rejected": "Si olvidaste tu 2FA ya perdiste tu cuenta para siempre, no hay nada que hacer, create una cuenta nueva y vuelve a pagar todos tus servicios desde cero."
    },
    {
        "prompt": "Explica en un parrafo que es Kubernetes para un gerente no tecnico.",
        "chosen": "Kubernetes es como un director de logistica automatizado para software: se asegura de que sus aplicaciones se ejecuten en los servidores adecuados, las repara automaticamente si fallan y agrega mas capacidad cuando hay alta demanda de clientes.",
        "rejected": "Kubernetes es un orquestador de contenedores CNCF escrito en Go que corre kubelet, kube-proxy, etcd como store Raft distribuido y coordina pods bajo namespaces usando cgroups y namespaces del kernel Linux."
    },
    {
        "prompt": "Como puedo resetear mi contraseña?",
        "chosen": "Para resetear su contraseña, haga clic en 'Olvidé mi contraseña' en la página de inicio de sesión y siga las instrucciones enviadas a su correo electrónico registrado.",
        "rejected": "No se puede. Si la olvidó, debe crear una cuenta nueva."
    },
    {
        "prompt": "Cuáles son los planes de precios disponibles?",
        "chosen": "Ofrecemos planes Básico, Pro y Empresarial. Cada uno con diferentes características y niveles de servicio. Visite nuestra página de precios para más detalles.",
        "rejected": "Los planes de precios cambian constantemente, es mejor que hable con un representante de ventas, pero no estoy seguro si estarán disponibles ahora."
    },
    {
        "prompt": "Necesito ayuda con la configuración de mi cuenta.",
        "chosen": "Puede encontrar guías detalladas de configuración en nuestra sección de ayuda o contactar a nuestro soporte técnico para asistencia personalizada.",
        "rejected": "La configuración es muy sencilla, si no puede hacerlo, probablemente es porque no está prestando atención."
    },
    {
        "prompt": "Cómo puedo contactar al soporte técnico?",
        "chosen": "Nuestro equipo de soporte técnico está disponible 24/7 a través de chat en vivo, correo electrónico (soporte@empresa.com) y teléfono (1-800-XXX-XXXX).",
        "rejected": "Solo se puede contactar a soporte por un formulario que está oculto en alguna parte de la página, buena suerte encontrándolo."
    },
    {
        "prompt": "Qué métodos de pago aceptan?",
        "chosen": "Aceptamos tarjetas de crédito (Visa, MasterCard, American Express), PayPal y transferencias bancarias para planes anuales.",
        "rejected": "Aceptamos cualquier cosa que tenga dinero. Pero el sistema a veces falla, así que intente varias veces."
    },
    {
        "prompt": "Puedo actualizar mi plan en cualquier momento?",
        "chosen": "Sí, puede actualizar su plan en cualquier momento desde su panel de control. Los cambios se aplicarán inmediatamente y se ajustará su facturación proporcionalmente.",
        "rejected": "No, una vez que elige un plan, se queda con él para siempre. Es una decisión importante."
    },
    {
        "prompt": "Cuál es su política de reembolso?",
        "chosen": "Ofrecemos un reembolso completo dentro de los primeros 30 días de su suscripción si no está satisfecho con nuestro servicio.",
        "rejected": "No hacemos reembolsos, todas las ventas son finales. Debería haber leído los términos y condiciones antes de pagar."
    },
    {
        "prompt": "Cómo funciona la integración con otras herramientas?",
        "chosen": "Nuestra plataforma ofrece integraciones nativas con las herramientas más populares. Consulte nuestra documentación para ver la lista completa y guías de configuración.",
        "rejected": "La integración es complicada y requiere conocimientos de programación avanzados. Contrate a un desarrollador si necesita ayuda."
    },
    {
        "prompt": "Puedo personalizar mi panel de control?",
        "chosen": "Sí, el panel de control es altamente personalizable. Puede organizar widgets, cambiar temas y configurar notificaciones según sus preferencias.",
        "rejected": "No, el panel de control tiene un diseño fijo. Acostúmbrese a él."
    },
    {
        "prompt": "Ofrecen alguna prueba gratuita?",
        "chosen": "Sí, ofrecemos una prueba gratuita de 14 días con acceso completo a todas las funciones del plan Pro. No se requiere tarjeta de crédito para registrarse.",
        "rejected": "Solo si se suscribe a nuestro plan anual y luego pide un reembolso. Pero no lo recomiendo, es mucho papeleo."
    },
    {
        "prompt": "Cómo puedo migrar mis datos?",
        "chosen": "Proporcionamos herramientas de migración guiada y un equipo de soporte dedicado para ayudarle con la transferencia de sus datos de forma segura y eficiente.",
        "rejected": "La migración de datos es su problema. Le sugiero que haga una copia de seguridad manual."
    },
    {
        "prompt": "Tienen documentación API?",
        "chosen": "Sí, nuestra API está completamente documentada con ejemplos de código y guías de uso. Puede encontrarla en nuestra sección de desarrolladores.",
        "rejected": "Nuestra API es secreta. Solo los desarrolladores aprobados pueden acceder a ella, y aún así, no hay documentación pública."
    },
    {
        "prompt": "Cómo puedo cambiar mi dirección de correo electrónico?",
        "chosen": "Puede actualizar su dirección de correo electrónico en la sección de 'Configuración de la cuenta' en su perfil. Se le pedirá que verifique la nueva dirección.",
        "rejected": "No puede cambiar su correo electrónico una vez que lo registra. Es el identificador único de su cuenta."
    },
    {
        "prompt": "Cuál es el tiempo de actividad garantizado (SLA)?",
        "chosen": "Garantizamos un tiempo de actividad del 99.9% para todos nuestros servicios, respaldado por nuestro acuerdo de nivel de servicio (SLA) disponible públicamente.",
        "rejected": "No garantizamos nada. Internet es impredecible, y nuestro servicio también."
    },
    {
        "prompt": "Puedo tener múltiples usuarios en una cuenta?",
        "chosen": "Sí, nuestros planes Pro y Empresarial permiten agregar múltiples usuarios con diferentes roles y permisos. Puede gestionarlos desde el panel de administración.",
        "rejected": "No, cada cuenta es personal e intransferible. Si necesita que alguien más use el servicio, debe crear otra cuenta."
    },
    {
        "prompt": "Cómo cancelo mi suscripción?",
        "chosen": "Para cancelar su suscripción, vaya a 'Configuración de la cuenta', luego a 'Suscripción' y haga clic en 'Cancelar Suscripción'.",
        "rejected": "Solo puede cancelar su suscripción si envía una carta certificada con 30 días de antelación. Es un proceso complicado."
    },
    {
        "prompt": "Dónde puedo encontrar tutoriales?",
        "chosen": "Tenemos una amplia biblioteca de tutoriales en video y guías paso a paso en nuestra sección de recursos educativos.",
        "rejected": "Tutoriales? No necesitamos tutoriales, nuestro producto es intuitivo. Si no entiende, es su problema."
    },
    {
        "prompt": "Es posible exportar mis datos?",
        "chosen": "Sí, puede exportar sus datos en varios formatos, como CSV o JSON, desde la sección de 'Gestión de Datos' de su panel de control.",
        "rejected": "Sus datos están en nuestros servidores. No podemos permitírselos. Son nuestra propiedad."
    },
    {
        "prompt": "Cómo puedo informar un error o problema técnico?",
        "chosen": "Puede informar errores a través de nuestro portal de soporte. Incluya detalles y capturas de pantalla para una resolución más rápida.",
        "rejected": "No nos importa si encuentra un error. Probablemente sea un problema de su lado. Reinicie su computadora."
    },
    {
        "prompt": "Cuál es la diferencia entre el plan Básico y el Pro?",
        "chosen": "El plan Pro incluye todas las características del Básico más almacenamiento adicional, soporte prioritario y acceso a funciones avanzadas como la API.",
        "rejected": "Básicamente, uno es barato y el otro es caro. Esa es la única diferencia real."
    },
    {
        "prompt": "Ofrecen soporte para desarrolladores?",
        "chosen": "Sí, ofrecemos foros de desarrolladores activos, documentación de API detallada y soporte técnico especializado para integraciones.",
        "rejected": "Los desarrolladores son responsables de entender cómo usar nuestra API por sí mismos. No ofrecemos apoyo especial."
    },
    {
        "prompt": "Cómo puedo cambiar mi zona horaria?",
        "chosen": "Puede ajustar su zona horaria en la configuración de su perfil para que todas las fechas y horas se muestren correctamente.",
        "rejected": "La zona horaria es global. No se puede cambiar. Viva con ello."
    },
    {
        "prompt": "Hay límites de uso en mi plan?",
        "chosen": "Cada plan tiene límites específicos de uso, como almacenamiento o número de transacciones. Consulte la descripción de su plan para más detalles.",
        "rejected": "Use el servicio tanto como quiera. No hay límites, a menos que lo use demasiado y se lo cortemos."
    },
    {
        "prompt": "Puedo pausar mi suscripción temporalmente?",
        "chosen": "Sí, ofrecemos la opción de pausar su suscripción por un período definido. Póngase en contacto con nuestro equipo de soporte para gestionar esto.",
        "rejected": "No, una vez activa, la suscripción no se puede pausar. Debe cancelar y luego volver a suscribirse si lo desea."
    },
    {
        "prompt": "Cómo obtengo una copia de mi historial de pagos?",
        "chosen": "Su historial de pagos completo está disponible en la sección de 'Facturación' de su cuenta, donde puede descargar los recibos.",
        "rejected": "Su historial de pagos es confidencial y no se puede acceder a él. Solo nosotros lo vemos."
    },
    {
        "prompt": "El servicio es compatible con dispositivos móviles?",
        "chosen": "Sí, nuestra plataforma es completamente responsive y compatible con todos los dispositivos móviles y tabletas, con aplicaciones dedicadas para iOS y Android.",
        "rejected": "Funciona en el navegador, pero no está optimizado para móviles. Úselo en una computadora de escritorio para la mejor experiencia."
    },
    {
        "prompt": "Cómo puedo dar de baja mi boletín de noticias?",
        "chosen": "Puede darse de baja de nuestro boletín haciendo clic en el enlace 'Cancelar suscripción' que se encuentra en la parte inferior de cualquier correo electrónico de marketing.",
        "rejected": "No puede darse de baja. Una vez que se inscribe, está en nuestra lista para siempre. Disfrute de nuestros correos electrónicos."
    },
    {
        "prompt": "Ofrecen capacitación o webinars?",
        "chosen": "Sí, regularmente realizamos webinars en vivo y ofrecemos sesiones de capacitación para ayudarle a aprovechar al máximo nuestro producto.",
        "rejected": "No tenemos tiempo para eso. El producto es bastante autoexplicativo."
    },
    {
        "prompt": "Qué es la autenticación de dos factores (2FA)?",
        "chosen": "2FA es una capa de seguridad adicional que requiere un segundo método de verificación además de su contraseña para acceder a su cuenta.",
        "rejected": "Es una molestia. Solo para gente paranoica. No la use si no quiere complicarse la vida."
    },
    {
        "prompt": "Cómo puedo reportar abuso?",
        "chosen": "Puede reportar cualquier tipo de abuso a través de nuestro formulario de contacto o enviando un correo electrónico a abuse@empresa.com. Investigaremos su caso diligentemente.",
        "rejected": "No nos ocupamos de los problemas de los usuarios. Resuélvalos usted mismo."
    },
    {
        "prompt": "Cuál es el tiempo de respuesta del soporte?",
        "chosen": "Nuestro equipo de soporte responde en menos de 24 horas para consultas estándar y en 1 hora para clientes con planes prioritarios.",
        "rejected": "Respondemos cuando podemos. Podría ser mañana, la próxima semana o nunca. Depende de la carga de trabajo."
    },
    {
        "prompt": "Hay alguna aplicación de escritorio disponible?",
        "chosen": "Actualmente no tenemos una aplicación de escritorio dedicada, pero nuestra plataforma web es totalmente funcional y accesible desde cualquier navegador.",
        "rejected": "No, solo existe la versión web. Si quiere una aplicación, tendrá que crearla usted mismo."
    },
    {
        "prompt": "Puedo obtener facturas con IVA?",
        "chosen": "Sí, todas nuestras facturas incluyen el desglose del IVA y puede descargarlas desde su panel de facturación para fines contables.",
        "rejected": "El IVA es complicado. No lo incluimos en las facturas, pero puede calcularlo usted mismo si lo necesita."
    },
    {
        "prompt": "Cómo se maneja la seguridad de mis datos?",
        "chosen": "Implementamos cifrado de extremo a extremo, monitoreo constante y auditorías de seguridad periódicas para proteger sus datos.",
        "rejected": "Sus datos están tan seguros como un casillero en una estación de tren. Esperamos que nadie los encuentre."
    },
    {
        "prompt": "Puedo usar mi propio dominio?",
        "chosen": "Sí, ofrecemos la opción de configurar un dominio personalizado para su sitio o servicio. Consulte nuestra guía de configuración de dominio.",
        "rejected": "No, solo puede usar el subdominio que le asignamos. No somos un registrador de dominios."
    },
    {
        "prompt": "Ofrecen descuentos para organizaciones sin fines de lucro?",
        "chosen": "Sí, tenemos un programa de descuentos especiales para organizaciones sin fines de lucro. Póngase en contacto con nuestro equipo de ventas para más información.",
        "rejected": "Todas las organizaciones pagan el mismo precio. No hacemos excepciones."
    },
    {
        "prompt": "Cómo puedo eliminar mi cuenta?",
        "chosen": "Puede solicitar la eliminación de su cuenta a través de la configuración de su perfil. Tenga en cuenta que este proceso es irreversible.",
        "rejected": "No puede eliminar su cuenta. Una vez que se une, se queda con nosotros para siempre. Es una política de retención."
    },
    {
        "prompt": "Qué sucede si excedo los límites de mi plan?",
        "chosen": "Si excede los límites de su plan, se le notificará y tendrá la opción de actualizar su plan o incurrir en cargos por uso excesivo.",
        "rejected": "Si excede los límites, le cortaremos el servicio sin previo aviso. Es su responsabilidad monitorear su uso."
    },
    {
        "prompt": "Puedo pagar anualmente en lugar de mensualmente?",
        "chosen": "Sí, ofrecemos opciones de pago anual que a menudo vienen con un descuento en comparación con los planes mensuales.",
        "rejected": "Solo aceptamos pagos mensuales. No nos interesan los pagos anuales."
    },
    {
        "prompt": "Hay alguna garantía de devolución de dinero?",
        "chosen": "Sí, ofrecemos una garantía de devolución de dinero de 30 días sin preguntas para todos los nuevos suscriptores.",
        "rejected": "No hay garantía de devolución de dinero. Lo que paga, pagado está."
    },
    {
        "prompt": "Cómo funciona la facturación automática?",
        "chosen": "Su suscripción se renueva automáticamente al final de cada ciclo de facturación. Puede gestionar esta configuración en su panel de facturación.",
        "rejected": "La facturación automática es un misterio. A veces funciona, a veces no. No lo entendemos del todo."
    },
    {
        "prompt": "Puedo obtener una demo del producto?",
        "chosen": "Sí, puede solicitar una demostración personalizada con uno de nuestros expertos en productos para conocer todas las funcionalidades.",
        "rejected": "No hacemos demostraciones. Vea los videos en YouTube si quiere ver cómo funciona."
    },
    {
        "prompt": "Cuáles son los requisitos del sistema para usar el servicio?",
        "chosen": "Nuestro servicio es basado en la web y compatible con los navegadores modernos. No se requieren instalaciones especiales de software.",
        "rejected": "Necesita el último sistema operativo, 16 GB de RAM, una tarjeta gráfica de alta gama y un procesador de 8 núcleos para que funcione."
    },
    {
        "prompt": "Hay foros de la comunidad o grupos de usuarios?",
        "chosen": "Sí, tenemos un foro activo de la comunidad donde los usuarios pueden hacer preguntas, compartir consejos y colaborar.",
        "rejected": "No nos gusta que los usuarios hablen entre ellos. Mantenemos las cosas centralizadas."
    },
    {
        "prompt": "Cómo puedo cambiar mi nombre de usuario?",
        "chosen": "Para cambiar su nombre de usuario, por favor contacte a nuestro equipo de soporte con su solicitud. Podemos ayudarle con esto manualmente.",
        "rejected": "No puede cambiar su nombre de usuario. Una vez establecido, es permanente."
    },
    {
        "prompt": "Ofrecen APIs para personalización avanzada?",
        "chosen": "Sí, nuestra API RESTful permite una personalización avanzada e integración profunda con sus sistemas existentes. Consulte nuestra documentación.",
        "rejected": "Nuestra API es solo para fines internos. No está diseñada para el uso de los clientes."
    },
    {
        "prompt": "Puedo transferir mi cuenta a otra persona?",
        "chosen": "Sí, la transferencia de cuentas es posible bajo ciertas condiciones. Por favor, póngase en contacto con nuestro equipo de soporte para obtener ayuda.",
        "rejected": "No, las cuentas son personales e intransferibles. Si alguien más quiere usar el servicio, debe crear una nueva cuenta."
    }
]

dpo_dataset = Dataset.from_list(preference_data)
print(f"Muestras de preferencia creadas: {len(dpo_dataset)}")
print("Estructura de la muestra 0:")
for k, v in dpo_dataset[0].items():
    print(f"  [{k}]: {v[:100]}...")

Muestras de preferencia creadas: 51
Estructura de la muestra 0:
  [prompt]: Como cancelo mi suscripcion y obtengo mi factura del mes en curso?...
  [chosen]: Para cancelar su suscripcion y descargar su factura: ingrese a su Perfil > Facturacion, seleccione C...
  [rejected]: No estoy totalmente seguro, pero creo que tienes que enviar un correo electronico a soporte tecnico ...


### Paso 5: Configuracion de Adaptadores LoRA para DPO

Inyectamos adaptadores LoRA mediante `peft`. Durante el entrenamiento DPO, el entrenador `DPOTrainer` evalua las probabilidades relativas alternando los adaptadores:


In [5]:
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 3,194,880 || all params: 2,617,536,768 || trainable%: 0.1221


### Paso 6: Configuracion y Entrenamiento con `DPOTrainer` (TRL)

Configuramos `DPOConfig`:
- `beta=0.1`: Ponderador de la divergencia KL contra el modelo base. Un $\beta$ mas alto mantiene al modelo mas cercano al modelo base original; un $\beta$ mas bajo permite una alineacion mas agresiva hacia las respuestas elegidas.
- `learning_rate=5e-6`: Tasa de aprendizaje muy conservadora (tipicamente 10x a 50x menor que en SFT) para evitar colapso de politicas:


In [9]:
import gc
import torch

# 1. Limpieza previa y configuracion del modelo (como en SFT)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.config.use_cache = False # Desactivar cache para el entrenamiento

# 2. Configuracion de DPO
dpo_config = DPOConfig(
    output_dir="./dpo_gemma_output",
    beta=0.1,
    learning_rate=5e-6,           # DPO requiere un LR menor que SFT para no colapsar
    lr_scheduler_type="linear",
    warmup_steps=5,
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    fp16=False,
    bf16=torch.cuda.is_available(),
    logging_steps=1,
    report_to="none"
)

# 3. Inicializar DPOTrainer (con processing_class)
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Con PEFT, ref_model=None es correcto
    train_dataset=dpo_dataset,
    args=dpo_config,
    processing_class=tokenizer  # <-- VITAL: para que formatee bien los prompts/respuestas
)

print("Iniciando alineacion de preferencias con DPO...")
dpo_trainer.train()

Adding EOS to train dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Dropping fully truncated examples from train dataset:   0%|          | 0/51 [00:00<?, ? examples/s]

Iniciando alineacion de preferencias con DPO...


Step,Training Loss
1,0.693147
2,0.693147
3,0.700487
4,0.708301
5,0.684413
6,0.673422
7,0.690138
8,0.679895
9,0.677790
10,0.693537


TrainOutput(global_step=130, training_loss=0.6070754152077895, metrics={'train_runtime': 337.4956, 'train_samples_per_second': 0.756, 'train_steps_per_second': 0.385, 'total_flos': 245536734643200.0, 'train_loss': 0.6070754152077895, 'epoch': 5.0})

### Paso 7: Evaluacion Cualitativa de la Alineacion

Generamos respuestas con el modelo alineado sobre las preguntas del dataset y observamos como ahora favorece el estilo constructivo, directo y libre de alucinaciones:


In [11]:
def generate_aligned_response(query, model_inst, tok_inst):
    # 1. Reactivar cache para una inferencia rapida y correcta
    model_inst.config.use_cache = True

    messages = [{"role": "user", "content": query}]
    prompt = tok_inst.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok_inst(prompt, return_tensors="pt").to(model_inst.device)

    with torch.no_grad():
        output = model_inst.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.3,
            do_sample=True,
            pad_token_id=tok_inst.pad_token_id,
            eos_token_id=tok_inst.eos_token_id
        )
    res = output[0][inputs["input_ids"].shape[1]:]
    return tok_inst.decode(res, skip_special_tokens=True).strip()

print("=== EVALUACION DE MODELO ALINEADO CON DPO ===\n")
test_q = "Explica en un parrafo que es Kubernetes para un gerente no tecnico."
print(f"Pregunta: {test_q}\n")
print(f"Respuesta Generada por el Modelo Alineado:\n{generate_aligned_response(test_q, model, tokenizer)}")

=== EVALUACION DE MODELO ALINEADO CON DPO ===

Pregunta: Explica en un parrafo que es Kubernetes para un gerente no tecnico.

Respuesta Generada por el Modelo Alineado:
ImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaImaIma


### Paso 8: Guardado de los Adaptadores Alineados por DPO

Guardamos el adaptador resultante de la optimizacion directa de preferencias:


In [12]:
dpo_output_dir = "./gemma_dpo_aligned_lora"
model.save_pretrained(dpo_output_dir)
tokenizer.save_pretrained(dpo_output_dir)
print(f"Adaptadores DPO exportados exitosamente a: {dpo_output_dir}")

Adaptadores DPO exportados exitosamente a: ./gemma_dpo_aligned_lora


### Paso 9: Limpieza de Memoria y Recursos

Liberacion de memoria VRAM:


In [9]:
del dpo_trainer, model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Recursos liberados exitosamente.")

Recursos liberados exitosamente.
